In [1]:
import pandas as pd

# Set pandas display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 100)        # Set the display width to 1000 characters
pd.set_option('display.max_colwidth', None) # Allow the full content of each column to be displayed

In [2]:
df_final_audit = pd.read_csv('./comprehensive_audit_final.csv')

In [3]:
import pandas as pd
import numpy as np
import pingouin as pg
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── 데이터 로드 ──────────────────────────────────────────
df = pd.read_csv('./sfscore_df.csv', encoding='utf-8-sig')
print(f"데이터: {len(df)}건")

# ── 분석 대상 변수 (ICC 0.4대 하락 변수) ─────────────────
target_vars = ['공복혈당', '당화혈색소', '허리둘레']

def extract_num(series):
    return pd.to_numeric(
        series.astype(str).str.extract(r'([-+]?\d*\.?\d+)', expand=False),
        errors='coerce'
    )

def calc_icc(orig, ext):
    """ICC(3,1) Absolute Agreement 산출"""
    temp = pd.DataFrame({'orig': orig, 'ext': ext}).dropna()
    if len(temp) < 2:
        return np.nan
    if np.allclose(temp['orig'], temp['ext'], atol=1e-9):
        return 1.000
    temp['ID'] = range(len(temp))
    df_long = pd.melt(temp, id_vars=['ID'],
                      value_vars=['orig', 'ext'],
                      var_name='Rater', value_name='Score')
    try:
        icc_res = pg.intraclass_corr(
            data=df_long, targets='ID', raters='Rater', ratings='Score'
        )
        return icc_res.set_index('Type').loc['ICC3', 'ICC']
    except:
        return np.nan

def bland_altman(orig, ext):
    """Bland-Altman 분석: 평균, 편향, LoA, t-test"""
    temp = pd.DataFrame({'orig': orig, 'ext': ext}).dropna()
    diff = temp['ext'] - temp['orig']
    mean_val = (temp['orig'] + temp['ext']) / 2
    bias = diff.mean()
    sd = diff.std()
    loa_upper = bias + 1.96 * sd
    loa_lower = bias - 1.96 * sd
    _, p_bias = stats.ttest_1samp(diff, 0)
    # 비례 편향: diff ~ mean 상관
    r_prop, p_prop = stats.pearsonr(mean_val, diff)
    return {
        'N': len(temp),
        'Bias(Mean Diff)': round(bias, 4),
        'SD(Diff)': round(sd, 4),
        'LoA Upper': round(loa_upper, 4),
        'LoA Lower': round(loa_lower, 4),
        'P(bias=0)': f"{'< .001' if p_bias < 0.001 else round(p_bias, 3)}",
        '체계적편향': '있음' if p_bias < 0.05 else '없음',
        'r(비례편향)': round(r_prop, 3),
        'P(비례편향)': f"{'< .001' if p_prop < 0.001 else round(p_prop, 3)}",
        '비례편향': '있음' if p_prop < 0.05 else '없음'
    }

# ── 분석 실행 ────────────────────────────────────────────
print("\n" + "=" * 80)
print("ICC 이상치 제거 전후 비교 및 Bland-Altman 분석")
print("=" * 80)

icc_results = []
ba_results = []

for var in target_vars:
    col_o = f'원본_{var}'
    col_e = f'추출_{var}'
    if col_o not in df.columns or col_e not in df.columns:
        print(f"[경고] {var} 컬럼 없음")
        continue

    orig = extract_num(df[col_o])
    ext  = extract_num(df[col_e])
    temp = pd.DataFrame({'orig': orig, 'ext': ext}).dropna()

    # ── ICC 전체 ─────────────────────────────────────────
    icc_all = calc_icc(temp['orig'], temp['ext'])

    # ── 이상치 제거 (IQR 기반, diff 기준) ────────────────
    diff = temp['ext'] - temp['orig']
    q1, q3 = diff.quantile(0.25), diff.quantile(0.75)
    iqr = q3 - q1
    mask = (diff >= q1 - 3 * iqr) & (diff <= q3 + 3 * iqr)
    temp_clean = temp[mask]
    n_removed = len(temp) - len(temp_clean)
    icc_clean = calc_icc(temp_clean['orig'], temp_clean['ext'])

    icc_results.append({
        '변수': var,
        'N(전체)': len(temp),
        'ICC(전체)': round(icc_all, 3),
        '제거건수': n_removed,
        '제거율(%)': round(n_removed/len(temp)*100, 2),
        'N(이상치제거)': len(temp_clean),
        'ICC(이상치제거)': round(icc_clean, 3),
        'ICC개선': round(icc_clean - icc_all, 3)
    })

    # ── Bland-Altman ──────────────────────────────────────
    ba = bland_altman(temp['orig'], temp['ext'])
    ba['변수'] = var
    ba_results.append(ba)

# ── 결과 출력 ────────────────────────────────────────────
print("\n▶ 1. ICC 이상치 제거 전후 비교 (IQR×3 기준)")
df_icc = pd.DataFrame(icc_results)
print(df_icc.to_string(index=False))

print("\n▶ 2. Bland-Altman 분석 결과")
df_ba = pd.DataFrame(ba_results)[[
    '변수', 'N', 'Bias(Mean Diff)', 'SD(Diff)',
    'LoA Upper', 'LoA Lower', 'P(bias=0)', '체계적편향',
    'r(비례편향)', 'P(비례편향)', '비례편향'
]]
print(df_ba.to_string(index=False))

# ── 저장 ─────────────────────────────────────────────────
df_icc.to_csv('icc_outlier_result.csv', index=False, encoding='utf-8-sig')
df_ba.to_csv('bland_altman_result.csv', index=False, encoding='utf-8-sig')
print("\n결과 저장: icc_outlier_result.csv / bland_altman_result.csv")

데이터: 10058건

ICC 이상치 제거 전후 비교 및 Bland-Altman 분석

▶ 1. ICC 이상치 제거 전후 비교 (IQR×3 기준)
   변수  N(전체)  ICC(전체)  제거건수  제거율(%)  N(이상치제거)  ICC(이상치제거)  ICC개선
 공복혈당  10047    0.525    57    0.57      9990         1.0  0.475
당화혈색소  10042    0.419     3    0.03     10039         1.0  0.581
 허리둘레  10047    0.443     9    0.09     10038         1.0  0.557

▶ 2. Bland-Altman 분석 결과
   변수     N  Bias(Mean Diff)  SD(Diff)  LoA Upper  LoA Lower P(bias=0) 체계적편향  r(비례편향) P(비례편향) 비례편향
 공복혈당 10047           0.7769   31.9400    63.3794   -61.8255     0.015    있음    0.558  < .001   있음
당화혈색소 10042           0.0196    1.4277     2.8180    -2.7787     0.168    없음    0.668  < .001   있음
 허리둘레 10047           0.2595   17.4017    34.3669   -33.8479     0.135    없음    0.640  < .001   있음

결과 저장: icc_outlier_result.csv / bland_altman_result.csv
